# Masking por bloque con cinco semillas, NSL-KDD

Compara el masking por bloque con el control. Se repite con las semillas 42, 1, 2, 3 y 4. La partición de los datos es fija, de modo que solo varía el azar del entrenamiento. Se entrena con KDDTrain+_20Percent y se evalúa con KDDTest+, que incluye tipos de ataque no vistos en entrenamiento.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]
BASE = Path.home() / ".cache/kagglehub/datasets/hassan06/nslkdd/versions/1"
df_train = pd.read_csv(BASE / "KDDTrain+_20Percent.txt", header=None, names=COL_NAMES).drop(columns=['difficulty'])
df_test = pd.read_csv(BASE / "KDDTest+.txt", header=None, names=COL_NAMES).drop(columns=['difficulty'])

df_train['is_attack'] = (df_train['label'] != 'normal').astype(int)
df_test['is_attack'] = (df_test['label'] != 'normal').astype(int)
df_train['_split'] = 'train'
df_test['_split'] = 'test'
df_all = pd.concat([df_train, df_test], ignore_index=True)

X_all = pd.get_dummies(df_all.drop(columns=['label', 'is_attack', '_split']), columns=['protocol_type', 'service', 'flag'])
is_train = (df_all['_split'] == 'train').to_numpy()

X_train_df = X_all[is_train].reset_index(drop=True)
X_test_df = X_all[~is_train].reset_index(drop=True)
y_train = df_train['is_attack'].to_numpy()
y_test = df_test['is_attack'].to_numpy()
print(f"X_train: {X_train_df.shape}, X_test: {X_test_df.shape}")

X_train: (25192, 119), X_test: (22544, 119)


In [3]:
scaler = MinMaxScaler().fit(X_train_df.to_numpy())
X_train_arr = scaler.transform(X_train_df.to_numpy())
X_test_arr = scaler.transform(X_test_df.to_numpy())
X_train_normal = X_train_arr[y_train == 0]

print(f"X_train: {X_train_arr.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test_arr.shape}")

X_train: (25192, 119), X_train_normal: (13449, 119), X_test: (22544, 119)


In [4]:
CAT_COLS = ['protocol_type', 'service', 'flag']
columns = list(X_all.columns)
group_of_column = []
for col in columns:
    matched = None
    for cat in CAT_COLS:
        if col.startswith(cat + '_'):
            matched = cat
            break
    group_of_column.append(matched if matched is not None else col)  # cada columna numérica forma su propio bloque

groups = sorted(set(group_of_column), key=lambda g: group_of_column.index(g))
group_index = {g: i for i, g in enumerate(groups)}
num_groups = len(groups)

group_matrix = np.zeros((num_groups, len(columns)), dtype=np.float32)
for col_idx, g in enumerate(group_of_column):
    group_matrix[group_index[g], col_idx] = 1.0

print(f"Columnas: {len(columns)}, bloques: {num_groups}")
print(f"Tamano de cada bloque: {group_matrix.sum(axis=1).astype(int).tolist()}")

Columnas: 119, bloques: 41
Tamano de cada bloque: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 67, 11]


In [5]:
class BlockMasking(layers.Layer):
    """Anula con probabilidad mask_rate bloques completos de columnas.

    group_matrix (num_groups, input_dim) indica a qué bloque pertenece cada columna.
    """
    def __init__(self, mask_rate, group_matrix, **kwargs):
        super().__init__(**kwargs)
        self.mask_rate = mask_rate
        self.group_matrix_np = group_matrix
        self.group_matrix = tf.constant(group_matrix, dtype=tf.float32)
        self.num_groups = group_matrix.shape[0]

    def call(self, inputs, training=None):
        if training:
            batch_size = tf.shape(inputs)[0]
            group_keep = tf.cast(
                tf.random.uniform((batch_size, self.num_groups)) >= self.mask_rate,
                tf.float32,
            )
            col_keep = tf.matmul(group_keep, self.group_matrix)
            return inputs * col_keep
        return inputs

    def get_config(self):
        config = super().get_config()
        config.update({'mask_rate': self.mask_rate})
        return config


def build_dae_block_masking(input_dim, hidden_units, mask_rate, group_matrix):
    inputs = keras.Input(shape=(input_dim,))
    x = BlockMasking(mask_rate, group_matrix)(inputs)
    for units in hidden_units[:-1]:
        x = layers.Dense(units, activation='relu')(x)
    x = layers.Dense(hidden_units[-1], activation='relu')(x)
    for units in reversed(hidden_units[:-1]):
        x = layers.Dense(units, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model


def reconstruction_error(model, X):
    return np.mean(np.square(X - model.predict(X, verbose=0)), axis=1)


HIDDEN_UNITS = [64, 32, 16]
MASK_RATES = [0.0, 0.1, 0.2, 0.3]  # 0.0 = control sin corrupción
SEEDS = [42, 1, 2, 3, 4]  # la partición de los datos se mantiene fija con SEED = 42

results = {}
for seed in SEEDS:
    for rate in MASK_RATES:
        tf.keras.utils.set_random_seed(seed)
        model = build_dae_block_masking(X_train_arr.shape[1], HIDDEN_UNITS, rate, group_matrix)
        history = model.fit(
            X_train_normal, X_train_normal,
            epochs=100,
            batch_size=64,
            validation_split=0.1,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
            verbose=0,
        )
        auc = roc_auc_score(y_test, reconstruction_error(model, X_test_arr))
        results[(seed, rate)] = auc
        print(f"seed={seed} mask_rate={rate}: epocas={len(history.history['loss'])}, ROC-AUC={auc:.4f}", flush=True)

seed=42 mask_rate=0.0: epocas=100, ROC-AUC=0.9406


seed=42 mask_rate=0.1: epocas=100, ROC-AUC=0.9359


seed=42 mask_rate=0.2: epocas=100, ROC-AUC=0.9431


seed=42 mask_rate=0.3: epocas=100, ROC-AUC=0.9327


seed=1 mask_rate=0.0: epocas=100, ROC-AUC=0.9256


...

seed=3 mask_rate=0.3: epocas=100, ROC-AUC=0.9457


seed=4 mask_rate=0.0: epocas=100, ROC-AUC=0.9329


seed=4 mask_rate=0.1: epocas=100, ROC-AUC=0.9223


seed=4 mask_rate=0.2: epocas=100, ROC-AUC=0.9521


seed=4 mask_rate=0.3: epocas=100, ROC-AUC=0.9220


In [6]:
import pandas as pd

df_res = pd.Series(results).unstack()  # filas: semilla, columnas: mask_rate
df_res.index.name = 'seed'
print("AUC por semilla y nivel de corrupcion:")
print(df_res.round(4).to_string())

print("\nMedia y desviacion tipica (5 semillas):")
for rate in MASK_RATES:
    v = df_res[rate]
    print(f"  mask_rate={rate}: media={v.mean():.4f}  std={v.std(ddof=1):.4f}  min={v.min():.4f}  max={v.max():.4f}")

print("\nDiferencia con el control de la misma semilla:")
for rate in MASK_RATES[1:]:
    d = df_res[rate] - df_res[0.0]
    print(f"  mask_rate={rate}: media={d.mean():+.4f}  std={d.std(ddof=1):.4f}  "
          f"semillas que mejoran al control={int((d > 0).sum())}/{len(d)}")

AUC por semilla y nivel de corrupcion:
         0.0     0.1     0.2     0.3
seed                                
1     0.9256  0.9473  0.9502  0.9455
2     0.9040  0.9502  0.9467  0.9427
3     0.9499  0.9356  0.9360  0.9457
4     0.9329  0.9223  0.9521  0.9220
42    0.9406  0.9359  0.9431  0.9327

Media y desviacion tipica (5 semillas):
  mask_rate=0.0: media=0.9306  std=0.0174  min=0.9040  max=0.9499
  mask_rate=0.1: media=0.9383  std=0.0111  min=0.9223  max=0.9502
  mask_rate=0.2: media=0.9456  std=0.0064  min=0.9360  max=0.9521
  mask_rate=0.3: media=0.9377  std=0.0102  min=0.9220  max=0.9457

Diferencia con el control de la misma semilla:
  mask_rate=0.1: media=+0.0077  std=0.0257  semillas que mejoran al control=2/5
  mask_rate=0.2: media=+0.0151  std=0.0216  semillas que mejoran al control=4/5
  mask_rate=0.3: media=+0.0071  std=0.0214  semillas que mejoran al control=2/5
